In [1]:
import os
import pandas as pd

In [2]:
INPUT_DIR = "../datasets"
OUTPUT_DIR = "../data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

### ЗАГРУЗКА ИСХОДНЫХ EXCEL-ФАЙЛОВ

In [3]:
def load_datasets(folder_path):
    datasets = {}

    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        if os.path.isdir(file_path):
            continue

        if filename.lower().endswith((".xlsx", ".xls")):
            key = os.path.splitext(filename)[0]
            datasets[key] = pd.read_excel(file_path)
            print(f"Загружен: {filename}")

    print(f"\nВсего загружено файлов: {len(datasets)}")

    return datasets


dfs = load_datasets(INPUT_DIR)

Загружен: Plan2025.xlsx
Загружен: RuHoliday.xlsx
Загружен: Sales_Report_2024-05-31.xlsx
Загружен: Sales_Report_2025-05-20.xlsx

Всего загружено файлов: 4


### ПОДГОТОВКА ПРОДАЖ

In [4]:
sales_2024 = dfs["Sales_Report_2024-05-31"].copy()
sales_2025 = dfs["Sales_Report_2025-05-20"].copy()


def prepare_sales(df):
    # Приводим исходную дату к datetime
    df["Дата"] = pd.to_datetime(
        df["Дата"],
        format="%d.%m.%Y %H:%M:%S"
    )

    df["Clean_Date"] = df["Дата"].dt.normalize()

    return df


sales_2024 = prepare_sales(sales_2024)
sales_2025 = prepare_sales(sales_2025)


# Объединяем продажи двух лет
sales_all = pd.concat(
    [sales_2024, sales_2025],
    ignore_index=True
)

### ПОДГОТОВКА КАЛЕНДАРЯ МАЯ

In [5]:
holidays = dfs["RuHoliday"].copy()

# Находим колонку с годом
year_col = next(
    col for col in holidays.columns
    if "Год" in col
)


def parse_holiday_string(value):
    """
    Преобразует строку вида:
    '1, 2, 8*, 9'
    в список (теперь сохраняет звездочку для рабочих предпраздничных дней):
    ['1', '2', '8*', '9']
    """
    if pd.isna(value):
        return []

    return [x.strip() for x in str(value).split(",") if x.strip()]


def create_may_calendar(year, holidays_list):
    calendar = []
    working_day_index = 0

    for day in range(1, 32):

        date = pd.Timestamp(year=year,month=5,day=day)

        is_weekend = date.weekday() >= 5

        # Проверяем точное совпадение с праздником (без звездочки)
        day_str = str(day)
        is_holiday = day_str in holidays_list

        # Если день есть в списке со звездочкой (например, '8*'), значит он рабочий (сокращенный)
        is_short_workday = f"{day}* " in holidays_list or f"{day}*" in holidays_list

        # Рабочий, если не выходной и не праздник, ИЛИ если это официально сокращенный день со звездочкой
        if (not is_weekend and not is_holiday) or is_short_workday:
            working_day_index += 1
            is_workday = 1
            wd_index = working_day_index
        else:
            is_workday = 0
            wd_index = None

        calendar.append({
            "Date": date,
            "Year": year,
            "Day_Of_Month": day,
            "Is_Workday": is_workday,
            "Working_Day_Index": wd_index
        })

    result = pd.DataFrame(calendar)
    result["Working_Day_Index"] = result["Working_Day_Index"].astype("Int64")

    return result


# Получаем праздники мая
holidays_2024 = parse_holiday_string(
    holidays.loc[
        holidays[year_col] == 2024,
        "Май"
    ].iloc[0]
)

holidays_2025 = parse_holiday_string(
    holidays.loc[
        holidays[year_col] == 2025,
        "Май"
    ].iloc[0]
)


calendar_2024 = create_may_calendar(
    2024,
    holidays_2024
)

calendar_2025 = create_may_calendar(
    2025,
    holidays_2025
)

calendar_may = pd.concat(
    [calendar_2024, calendar_2025],
    ignore_index=True
)

print(f'Выходные дни 2024 года: {holidays_2024}')

print(f'Выходные дни 2025 года: {holidays_2025}')

['1', '2', '3', '4', '8', '9', '10', '11', '17', '18', '24', '25', '31']


### ПОДГОТОВКА ПЛАНА 2025

In [6]:
plan_2025 = dfs["Plan2025"].copy()

# Находим колонку с планом в млн
plan_col = next(
    col for col in plan_2025.columns
    if "млн" in col
)

# Переводим млн рублей → рубли
plan_2025["План_Рублей"] = (
    plan_2025[plan_col] * 1_000_000
)


### СПРАВОЧНИКИ ДЛЯ POWER BI

In [7]:

# Справочник подразделений
dim_subdivisions = (
    sales_all[["Подразделение"]]
    .dropna()
    .drop_duplicates()
    .sort_values("Подразделение")
    .reset_index(drop=True)
)

# Справочник товарных категорий
dim_categories = (
    sales_all[["Номенклатура.Товарная категория"]]
    .dropna()
    .drop_duplicates()
    .sort_values("Номенклатура.Товарная категория")
    .reset_index(drop=True)
)

### СОХРАНЕНИЕ ДАННЫХ ДЛЯ POWER BI

In [8]:
sales_path = os.path.join(OUTPUT_DIR, "Sales_Clean.csv")
plan_path = os.path.join(OUTPUT_DIR, "Plan_2025_Clean.csv")
calendar_path = os.path.join(OUTPUT_DIR, "Calendar_May.csv")
subdivisions_path = os.path.join(OUTPUT_DIR, "Dim_Subdivisions.csv")
categories_path = os.path.join(OUTPUT_DIR, "Dim_Categories.csv")


sales_all.to_csv(
    sales_path,
    index=False,
    encoding="utf-8-sig"
)

plan_2025.to_csv(
    plan_path,
    index=False,
    encoding="utf-8-sig"
)

calendar_may.to_csv(
    calendar_path,
    index=False,
    encoding="utf-8-sig"
)

dim_subdivisions.to_csv(
    subdivisions_path,
    index=False,
    encoding="utf-8-sig"
)

dim_categories.to_csv(
    categories_path,
    index=False,
    encoding="utf-8-sig"
)


print(f"\nФайлы сохранены в папке: '{OUTPUT_DIR}'.")
print("\n" + "=" * 60)
print("ETL ЗАВЕРШЁН")
print("=" * 60)

print("\nФайлы готовы для загрузки в Power BI.")



Файлы сохранены в папке: '../data'.

ETL ЗАВЕРШЁН

Файлы готовы для загрузки в Power BI.
